# 完整商业方案

## 现在把第 1 天的项目提升到下一阶段

### 商业挑战：

做一个能为公司生成宣传册的产品，面向潜在客户、投资人与求职者。

输入是公司名称及其主站。

本 notebook 末尾有真实商业应用示例。

记住：有问题或想法随时找我！

In [ ]:
# 导入
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [ ]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

In [ ]:
links = fetch_website_links("https://edwarddonner.com")
links

## 第一步：让 GPT-5-nano 判断哪些链接相关

### 调用 gpt-5-nano 阅读网页链接，并以结构化 JSON 回复。  
它应判断相关链接，并把 "/about" 这类相对链接补成 "https://company.com/about"。  
我们使用「单次示例提示（one-shot prompting）」，在提示里给出期望回复示例。

这很适合 LLM：需要细腻理解。若不用 LLM、纯靠解析网页硬编码，会非常难！

补充：还有更高级的「Structured Outputs」，强制模型按规范回复。第 8 周自主 Agent 项目会讲。

In [ ]:
# ask gpt to get what a useful brochure should include

link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [ ]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages etc.
you decide which links are most relevant to include in a brochure about the company.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [ ]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [ ]:
print(get_links_user_prompt("https://edwarddonner.com"))

In [ ]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [ ]:
select_relevant_links("https://edwarddonner.com")

In [ ]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [ ]:
select_relevant_links("https://edwarddonner.com")

In [ ]:
select_relevant_links("https://huggingface.co")

## 第二步：生成宣传册！

把全部细节组装进另一条提示，再交给 GPT-5-nano

In [ ]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [ ]:
print(fetch_page_and_all_relevant_links("https://www.legisys.ai"))

In [ ]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [ ]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [ ]:
get_brochure_user_prompt("Legisys", "https://www.legisys.ai")

In [ ]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [ ]:
create_brochure("Legisys", "https://www.legisys.ai")

## 最后——一个小改进

只需小改，就能让结果从 OpenAI 流式返回，
带上熟悉的打字机动画效果

In [ ]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
stream_brochure("Legisys", "https://www.legisys.ai")

## 面向受众的宣传册生成

In [ ]:
target_audience = ["B2B", "B2C", "Enterprise", "SMB", "Technical", "non-technical"]
role = ["Executives", "Legal", "Sales", "Investors", "Customers", "Prospective Employees"]

audience_aware_brochure_system_prompt = """
You are an AI assistant that analyzes the contents of several relevant pages from a company website
and creates a concise, professional brochure.

The user will provide two key inputs:
1. Target audience (e.g., customers, investors, recruits)
2. Role or perspective (e.g., buyer, decision-maker, engineer, executive, job seeker)

You must adapt the brochure’s:
- Tone
- Structure
- Emphasis
- Level of detail

based explicitly on the provided target audience and role.

Your responsibilities:

1. Extract and synthesize brochure-worthy content from the website, including:
   - Company overview and positioning
   - Product or service summary
   - Core value proposition
   - Differentiators and strengths

2. Adjust emphasis by audience:
   - **Customers** → focus on problems solved, benefits, usability, outcomes
   - **Investors** → focus on vision, market, differentiation, traction, leadership
   - **Recruits** → focus on culture, mission, team, growth, and career opportunities

3. Adjust framing by role:
   - Highlight what matters most to that role
   - Avoid unnecessary details irrelevant to that role
   - Use language appropriate for the role’s level of technical or business familiarity

4. Include the following sections **only if relevant and supported by the website content**:
   - Company culture and values
   - Customers or partners
   - Careers or job opportunities

5. Exclude or de-prioritize:
   - Navigation menus and UI elements
   - Legal, policy, or compliance pages
   - Blog posts and technical documentation unless directly relevant
   - Repetitive or low-value marketing language

6. Do NOT invent facts or claims not present on the website.
   - If important brochure elements are missing, omit them quietly rather than guessing.

Output requirements:
- Do NOT use code blocks
- Keep the brochure concise, polished, and audience-appropriate
"""

In [ ]:
def get_audience_aware_brochure_user_prompt(company_name, url, target_audience, role):
    user_prompt = f"""
You are analyzing a company called: {company_name}

Target audience: {target_audience}
Role / perspective: {role}

You are provided with the contents of the company’s landing page and other relevant pages.
Using this information, create a concise, professional brochure tailored specifically
to the given target audience and role.

Adapt the tone, structure, and emphasis to what matters most to this audience and role.

Respond in markdown without code blocks.
Do not invent information that is not present in the provided content.
Only include sections (e.g., culture, customers, careers) if supported by the content.

Below is the website content:
"""

    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000]  # Truncate if more than 5,000 characters
    return user_prompt

In [ ]:
def stream_audience_aware_brochure(company_name, url, target_audience, role):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": audience_aware_brochure_system_prompt},
            {"role": "user", "content": get_audience_aware_brochure_user_prompt(company_name, url, target_audience, role)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

## 思路：用户从我们给出的列表选择目标受众与角色，再据此生成宣传册

In [ ]:
stream_audience_aware_brochure("Legisys", "https://www.legisys.ai", target_audience[0], role[1])

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">商业应用</h2>
            <span style="color:#181;">本练习扩展了第 1 天代码：多次调用 LLM 并生成文档。

这或许是 Agentic AI 设计模式的第一个例子——组合多次 LLM 调用。第 2 周会更多；第 8 周构建完全自主 Agent 时会大规模回归。

这种方式生成内容是最常见用例之一。与摘要一样，可应用于任何业务领域：营销文案、从规格生成产品教程、个性化邮件等。探索如何应用到你的业务，并做个概念验证。也看看 community-contributions 里同学们的作品——宝藏很多！</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">进入第 2 周之前（那周超好玩）</h2>
            <span style="color:#900;">请完成 week1 EXERCISE notebook 作为第 1 周末挑战。这能给你前沿 API 的关键练习，并为第 2 周做好准备。</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">提醒：3 个实用资源</h2>
            <span style="color:#f71;">1. 课程资源见 <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">这里</a>。<br/>
            2. 我的 LinkedIn 在<a href="https://www.linkedin.com/in/eddonner/">这里</a>，很乐意与学员连接！<br/>
            3. 我在尝试 X/Twitter：<a href="https://x.com/edwarddonner">@edwarddonner</a>，希望大家教我怎么玩……  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">最后！有一个特别请求</h2>
            <span style="color:#090;">
                编辑告诉我：学员在 Udemy 评分影响巨大——这是 Udemy 决定是否推荐课程的主要方式之一。若你能花一分钟评分，我将非常感激！无论如何，需要帮助随时邮件 ed@edwarddonner.com。
            </span>
        </td>
    </tr>
</table>